# Tarang receipts · PaddleOCR-VL retry for repetition loops

The full run (`paddleocr_vl.ipynb`) read 467 photos; **11 hit the 768-token cap** because the model got stuck repeating a
line (in one case it invented amounts, `Total Cost: ₹3,997,249`). This notebook re-reads **only those photos** with a mild
`repetition_penalty=1.05`, everything else unchanged. It writes a separate file and backend name, so the benchmark can report the first
run and the retry side by side instead of silently replacing failures.

Kept mild on purpose: receipts legitimately repeat numbers (rate and amount, subtotal and total), and a strong penalty would distort them.

**Setup:** attach `tarang-receipts` · *GPU T4* (one is enough) · *Internet On* · fresh session · *Run All* (~2 minutes).
**Output:** `/kaggle/working/paddleocr_vl_retry.jsonl` → `data/interim/vlm/`.

In [ ]:
# Settings
RETRY_FILES = [
    "IMG-20260614-WA0186.jpg",
    "IMG-20260629-WA0170.jpg",
    "IMG-20260712-WA0059.jpg",
    "IMG-20260723-WA0041.jpg",
    "IMG_20260629_170106.jpg",
    "IMG_20260710_195933.jpg",
    "IMG_20260714_123934.jpg",
    "IMG_20260718_081516.jpg",
    "IMG_20260718_111930.jpg",
    "IMG_20260806_142652.jpg",
    "IMG_20260807_213207.jpg"
]
REPETITION_PENALTY = 1.05
BATCH_SIZE = 4
DTYPE = "auto"            # float32 on a T4 (same as the full run)
MAX_NEW_TOKENS = 768      # same cap as the full run
TRANSFORMERS_VERSION = "5.19.0"

In [ ]:
import subprocess, sys

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"transformers=={TRANSFORMERS_VERSION}", "accelerate"],
                   capture_output=True, text=True)
print("pip:", "ok" if r.returncode == 0 else r.stderr[-1500:])
check = subprocess.run([sys.executable, "-c", "import torch, transformers; from transformers import PaddleOCRVLForConditionalGeneration; "
                        "print(torch.__version__, torch.cuda.device_count(), 'GPUs', transformers.__version__)"],
                       capture_output=True, text=True)
if check.returncode:
    print(check.stderr[-1500:])
    raise RuntimeError("torch/transformers broken in this session. Run -> Factory reset, then Run All (do not install Paddle).")
print("torch, GPUs, transformers:", check.stdout.strip())

In [ ]:
import glob, json, os, time
from pathlib import Path

import pandas as pd

jpgs = sorted(glob.glob("/kaggle/input/**/*.jpg", recursive=True))
assert jpgs, "No .jpg files under /kaggle/input. Attach the tarang-receipts dataset (Add Input -> Datasets)."
audit_csv = next(iter(glob.glob("/kaggle/input/**/image_audit.csv", recursive=True)), "")
WORK = Path("/kaggle/working")
OUT_JSONL = WORK / "paddleocr_vl_retry.jsonl"
print(f"{len(jpgs)} photos | audit: {audit_csv or 'missing (no rotation fix)'}")

In [ ]:
WORKER = r"""
import argparse, json, sys, time
from pathlib import Path
import pandas as pd, torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

ap = argparse.ArgumentParser()
ap.add_argument("--files"); ap.add_argument("--out"); ap.add_argument("--audit", default="")
ap.add_argument("--batch", type=int); ap.add_argument("--dtype"); ap.add_argument("--max-new", type=int)
ap.add_argument("--probe", action="store_true"); ap.add_argument("--rep-penalty", type=float, default=1.0)
a = ap.parse_args()

files = json.loads(Path(a.files).read_text())
rotation = {}
if a.audit:
    au = pd.read_csv(a.audit, dtype=str, keep_default_na=False)
    rotation = {r.image_file: int(float(r.rotation)) for r in au.itertuples() if r.rotation not in ("", "nan")}

cap = torch.cuda.get_device_capability(0)
dtype = (torch.bfloat16 if cap >= (8, 0) else torch.float32) if a.dtype == "auto" else getattr(torch, a.dtype)
MODEL_ID = "PaddlePaddle/PaddleOCR-VL"
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=dtype).to("cuda").eval()
model.generation_config.use_cache = True     # the published config says False: that made generation ~50x slower
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"
backend = f"paddleocr-vl-0.9b-hf-native-{str(dtype).split('.')[-1]}" + (f"-rp{a.rep_penalty}" if a.rep_penalty != 1.0 else "")
pad = processor.tokenizer.pad_token_id
print(f"[{torch.cuda.get_device_name(0)}] {type(model).__name__} {dtype} {backend}", flush=True)

def read(names):
    ims, angles = [], []
    for n in names:
        ang = rotation.get(Path(n).name, 0)
        ims.append(Image.open(n).convert("RGB").rotate(ang, expand=True)); angles.append(ang)
    batch = [[{"role": "user", "content": [{"type": "image", "image": im}, {"type": "text", "text": "OCR:"}]}] for im in ims]
    inputs = processor.apply_chat_template(batch, add_generation_prompt=True, tokenize=True, return_dict=True,
                                           return_tensors="pt", processor_kwargs={"padding": True}).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=a.max_new, do_sample=False, use_cache=True, repetition_penalty=a.rep_penalty)
    new = out[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(new, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return texts, angles, [int((t != pad).sum()) for t in new]

if a.probe:
    read(files[:1])                                    # warm-up
    t0 = time.perf_counter(); texts, _, k = read(files[:2]); dt = time.perf_counter() - t0
    print(f"PROBE {dt / 2:.1f}s/photo, {sum(k) / dt:.1f} tokens/s", flush=True)
    print(texts[0][:400], flush=True)
    sys.exit(0)

out_path = Path(a.out)
done = set()
if out_path.exists():
    for line in out_path.read_text().splitlines():
        r = json.loads(line)
        if not r["error"] and r["backend"] == backend:
            done.add(r["file"])
todo = [f for f in files if Path(f).name not in done]
print(f"{len(done)} done, {len(todo)} to go", flush=True)

read(todo[:1]) if todo else None                       # warm-up, not timed
for i in range(0, len(todo), a.batch):
    names = todo[i:i + a.batch]
    t0 = time.perf_counter()
    try:
        texts, angles, k = read(names); errs = [""] * len(names)
    except Exception as e:
        torch.cuda.empty_cache()
        texts, angles, k = [""] * len(names), [None] * len(names), [0] * len(names)
        errs = [f"{type(e).__name__}: {e}"[:500]] * len(names)
    dt = time.perf_counter() - t0
    with out_path.open("a") as f:
        for n, t, ang, kk, err in zip(names, texts, angles, k, errs):
            f.write(json.dumps({"file": Path(n).name, "backend": backend, "text": t, "rotation_applied": ang,
                                "new_tokens": kk, "batch_size": len(names), "batch_seconds": round(dt, 3),
                                "seconds": round(dt / len(names), 3), "error": err}, ensure_ascii=False) + "\n")
    print(f"PROGRESS {min(i + a.batch, len(todo))}/{len(todo)} {dt / len(names):.2f}s/photo", flush=True)
print("DONE", flush=True)
"""
(WORK / "worker.py").write_text(WORKER)

def worker_cmd(gpu, files, out, probe=False):
    lst = WORK / f"files_gpu{gpu}.json"
    lst.write_text(json.dumps(files))
    cmd = [sys.executable, str(WORK / "worker.py"), "--files", str(lst), "--out", str(out), "--audit", audit_csv,
           "--batch", str(BATCH_SIZE), "--dtype", DTYPE, "--max-new", str(MAX_NEW_TOKENS), "--rep-penalty", str(REPETITION_PENALTY)] + (["--probe"] if probe else [])
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return cmd, env

In [ ]:
files = [p for p in jpgs if Path(p).name in set(RETRY_FILES)]
assert len(files) == len(RETRY_FILES), set(RETRY_FILES) - {Path(p).name for p in files}
cmd, env = worker_cmd(0, files, OUT_JSONL)
r = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode:
    print(r.stderr[-3000:])
    raise RuntimeError("worker failed; send this output")

In [ ]:
out = pd.read_json(OUT_JSONL, lines=True).drop_duplicates("file", keep="last")
print(f"{len(out)} photos, errors {(out.error != '').sum()}, still hit the cap: {(out.new_tokens >= MAX_NEW_TOKENS).sum()}")
for _, r in out.iterrows():
    print(f"\n===== {r.file}  ({r.new_tokens} tokens)\n{r.text[:700]}")